# DLP Detection Model — GBT vs CatBoost Comparison
Loads both trained models and compares them side by side across
evaluation metrics, confusion matrices, feature importance and threshold tuning.

In [0]:
%pip install catboost shap numpy==1.26.4 --quiet

In [0]:
import os, pickle
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score,classification_report, confusion_matrix,precision_recall_curve, roc_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split

## 1. Load Data

In [0]:
df_features_clean = spark.table("workspace.default.dlp_features_clean")
df = df_features_clean.toPandas()

print(f"Total rows: {len(df)}")
print(df["Risk_Label"].value_counts())

## 2. Define Feature Columns

In [0]:
LABEL_COL = "Risk_Label"

CATEGORICAL_COLS = [
    "ActionType",
    "file_extension",
    "Position",
]

NUMERICAL_COLS = [
    "is_high_risk_extension",
    "file_name_has_sensitive_keyword",
    "double_extension_check",
    "is_risky_target_domain",
    "is_first_time_domain",
    "is_after_hours",
    "day_of_week",
    "is_monday_or_friday",
    "is_high_risk_position",
    "user_upload_count_24h",
    "user_unique_domains_7d",
]

FEATURE_COLS = CATEGORICAL_COLS + NUMERICAL_COLS

## 3. Reproduce Test Split


In [0]:
X = df[FEATURE_COLS].copy()
y = df[LABEL_COL]

# GBT split (2-way) — same random_state=42 and stratify as Model_Training_GBT
X_train_gbt, X_test_gbt, y_train_gbt, y_test_gbt = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# CatBoost split (3-way) — same random_state=42 and stratify as Model_Training_CatBoost
X_train_val_cb, X_test_cb, y_train_val_cb, y_test_cb = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# CatBoost needs string categoricals
for c in CATEGORICAL_COLS:
    X_test_cb[c] = X_test_cb[c].astype(str)

print(f"GBT test set     : {len(X_test_gbt)} rows")
print(f"CatBoost test set: {len(X_test_cb)} rows")

## 4. Load Both Models

In [0]:
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
repo_root = os.path.dirname(notebook_path)

# Load GBT pipeline (sklearn)
with open(f"/Workspace{repo_root}/dlp_gbt_v1.pkl", "rb") as f:
    gbt_pipeline = pickle.load(f)

# Load CatBoost model
catboost_model = CatBoostClassifier()
catboost_model.load_model(f"/Workspace{repo_root}/dlp_catboost_v1.cbm")

print("Both models loaded.")

## 5. Generate Predictions

In [0]:
# GBT
gbt_pred      = gbt_pipeline.predict(X_test_gbt)
gbt_prob      = gbt_pipeline.predict_proba(X_test_gbt)[:, 1]

# CatBoost
cb_pred       = catboost_model.predict(X_test_cb)
cb_prob       = catboost_model.predict_proba(X_test_cb)[:, 1]


## 6. Metrics Comparison

In [0]:
metrics = {
    "AUC-ROC":             [roc_auc_score(y_test_gbt, gbt_prob),          roc_auc_score(y_test_cb, cb_prob)],
    "AUC-PR":              [average_precision_score(y_test_gbt, gbt_prob), average_precision_score(y_test_cb, cb_prob)],
}

metrics_df = pd.DataFrame(metrics, index=["GBT", "CatBoost"]).T
print(metrics_df.to_string())
print()

print(" GBT ")
print(classification_report(y_test_gbt, gbt_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test_gbt, gbt_pred))


In [0]:
print(" CatBoost ")
print(classification_report(y_test_cb, cb_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test_cb, cb_pred))

## 7. Feature Importance Comparison

In [0]:
gbt_importance = gbt_pipeline.named_steps["classifier"].feature_importances_
cb_importance  = catboost_model.get_feature_importance()

importance_df = (
    pd.DataFrame({
        "feature":   FEATURE_COLS,
        "GBT":       gbt_importance,
        "CatBoost":  cb_importance / 100  # CatBoost returns percentages, normalise to 0-1
    })
    .sort_values("GBT", ascending=False)
    .reset_index(drop=True)
)
print(importance_df.to_string(index=False))

## 8. Threshold Tuning Comparison
Finds the optimal F1 threshold for each model and shows
how results change compared to the default 0.5 threshold.

In [0]:
def tune_threshold(y_true, y_prob, model_name):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores      = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx       = f1_scores.argmax()
    best_threshold = thresholds[best_idx]
    y_pred_tuned   = (y_prob >= best_threshold).astype(int)

    print(f"── {model_name} ──────────────────────────")
    print(f"Best threshold : {best_threshold:.4f}")
    print(f"Precision      : {precisions[best_idx]:.4f}")
    print(f"Recall         : {recalls[best_idx]:.4f}")
    print(f"F1             : {f1_scores[best_idx]:.4f}")
    print("Confusion Matrix at tuned threshold:")
    print(confusion_matrix(y_true, y_pred_tuned))
    print()
    return best_threshold

gbt_threshold = tune_threshold(y_test_gbt, gbt_prob, "GBT")
cb_threshold  = tune_threshold(y_test_cb,  cb_prob,  "CatBoost")

## 9. Final Recommendation

In [0]:
gbt_cm = confusion_matrix(y_test_gbt, gbt_pred)
cb_cm  = confusion_matrix(y_test_cb,  cb_pred)

gbt_tn, gbt_fp, gbt_fn, gbt_tp = gbt_cm.ravel()
cb_tn,  cb_fp,  cb_fn,  cb_tp  = cb_cm.ravel()

gbt_precision = precision_score(y_test_gbt, gbt_pred)
gbt_recall    = recall_score(y_test_gbt, gbt_pred)
cb_precision  = precision_score(y_test_cb,  cb_pred)
cb_recall     = recall_score(y_test_cb,  cb_pred)

gbt_auc_roc = roc_auc_score(y_test_gbt, gbt_prob)
cb_auc_roc  = roc_auc_score(y_test_cb,  cb_prob)
gbt_auc_pr  = average_precision_score(y_test_gbt, gbt_prob)
cb_auc_pr   = average_precision_score(y_test_cb,  cb_prob)

summary = pd.DataFrame({
    "Metric": ["AUC-ROC", "AUC-PR", "Precision (class 1)", "Recall (class 1)", "False Positives", "False Negatives"],
    "GBT": [
        f"{gbt_auc_roc:.4f}",
        f"{gbt_auc_pr:.4f}",
        f"{gbt_precision:.4f}",
        f"{gbt_recall:.4f}",
        str(gbt_fp),
        str(gbt_fn)
    ],
    "CatBoost": [
        f"{cb_auc_roc:.4f}",
        f"{cb_auc_pr:.4f}",
        f"{cb_precision:.4f}",
        f"{cb_recall:.4f}",
        str(cb_fp),
        str(cb_fn)
    ],
    "Winner": [
        "CatBoost" if cb_auc_roc > gbt_auc_roc else "GBT",
        "CatBoost" if cb_auc_pr  > gbt_auc_pr  else "GBT",
        "CatBoost" if cb_precision > gbt_precision else "GBT",
        "CatBoost" if cb_recall  > gbt_recall   else "GBT",
        "CatBoost" if cb_fp < gbt_fp else "GBT",
        "CatBoost" if cb_fn < gbt_fn else "GBT",
    ]
})
print(summary.to_string(index=False))